In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, 2000).clip(5, 5000),
    'hour': np.random.normal(14, 4, 2000).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, 2000),
    'tx_per_day': np.random.poisson(3, 2000),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, 100),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], 100),
    'is_electronics': np.random.binomial(1, 0.7, 100),
    'tx_per_day': np.random.poisson(8, 100),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("---")
print("Gotowe! Model został wytrenowany i zapisany.")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

---
Gotowe! Model został wytrenowany i zapisany.


In [7]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

@app.get("/health")
async def health_check():
    return {"status": "ok", "model_loaded": True}

@app.post("/score")
async def score_transaction(td: Transaction):
    data = np.array([[td.amount, td.hour, td.is_electronics, td.tx_per_day]])
    prediction = model.predict(data)[0]
    probability = model.predict_proba(data)[0][1]
    return {
        "is_fraud": bool(prediction),
        "fraud_probability": float(probability)
    }

Overwriting fraud_api.py


In [9]:
import requests

# Test normalna
r = requests.post("http://localhost:8002/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

# Test podejrzana
r_fraud = requests.post("http://localhost:8002/score",
    json={"amount": 5500, "hour": 3, "is_electronics": 1, "tx_per_day": 12})
print("Podejrzana:", r_fraud.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 1.0}


In [10]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://jupyter:8002/score"

for message in consumer:
    tx = message.value
    
    # 1. Wyciągnij cechy
    # Godzinę wyciągamy z timestampa (format ISO: YYYY-MM-DDTHH:MM:SS)
    dt = datetime.fromisoformat(tx['timestamp'])
    
    features = {
        "amount": tx['amount'],
        "hour": dt.hour,
        "is_electronics": 1 if tx['category'] == 'electronics' else 0,
        "tx_per_day": 5
    }
    
    # 2. Odpytaj API
    try:
        response = requests.post(API_URL, json=features)
        result = response.json()
        
        # 3. Jeśli fraud, wyślij alert
        if result.get('is_fraud'):
            alert = {
                "tx_id": tx['tx_id'],
                "status": "ALERT",
                "probability": result['fraud_probability'],
                "details": tx
            }
            alert_producer.send('alerts', alert)
            print(f"!!! ALERT !!! Wykryto oszustwo: {tx['tx_id']} (Prob: {result['fraud_probability']})")
        else:
            print(f"Transakcja OK: {tx['tx_id']}")
            
    except Exception as e:
        print(f"Błąd API: {e}")

Overwriting ml_consumer.py


In [8]:
import requests
try:
    response = requests.get("http://localhost:8001/health")
    print("API żyje i ma się dobrze:", response.json())
except:
    print("API jednak nie działa.")

API żyje i ma się dobrze: {'detail': 'Not Found'}


In [11]:
git status


SyntaxError: invalid syntax (339777499.py, line 1)